In [3]:
import ollama
import psutil
import os

def summarize_and_measure(text, model_name="phi3"):
    print(f"Loading {model_name} and generating summary...\n")
    
    prompt = f"Please provide a concise summary of the following text:\n\n{text}"
    
    # Record memory before generation
    process = psutil.Process(os.getpid())
    memory_before = process.memory_info().rss / (1024 * 1024) # Convert to MB

    try:
        # Generate the response
        response = ollama.chat(model=model_name, messages=[
            {'role': 'user', 'content': prompt}
        ])
        
        # Record memory after generation
        memory_after = process.memory_info().rss / (1024 * 1024)
        
        # --- EXTRACT METRICS ---
        # Ollama returns durations in nanoseconds, so we divide by 1 billion to get seconds (1e9)
        total_time = response['total_duration'] / 1e9
        load_time = response.get('load_duration', 0) / 1e9
        
        input_tokens = response.get('prompt_eval_count', 0)
        prompt_eval_time = response.get('prompt_eval_duration', 0) / 1e9 # Time to First Token
        
        output_tokens = response.get('eval_count', 0)
        generation_time = response.get('eval_duration', 0) / 1e9
        
        # Calculate Tokens Per Second (TPS)
        tps = output_tokens / generation_time if generation_time > 0 else 0
        
        print("--- SUMMARY ---")
        print(response['message']['content'])
        print("\n" + "="*40)
        print("📊 PERFORMANCE METRICS 📊")
        print("="*40)
        print(f"Total Time Taken:      {total_time:.2f} seconds")
        print(f"Model Load Time:       {load_time:.2f} seconds")
        print(f"Time to First Token:   {prompt_eval_time:.2f} seconds")
        print("-" * 40)
        print(f"Input Length:          {input_tokens} tokens")
        print(f"Output Length:         {output_tokens} tokens")
        print(f"Generation Speed:      {tps:.2f} tokens/second")
        print("-" * 40)
        print(f"Python RAM Increase:   {memory_after - memory_before:.2f} MB")
        print("="*40)
        
    except Exception as e:
        print(f"An error occurred: {e}")

# Run it!
sample_text = """
The James Webb Space Telescope (JWST) is a space telescope designed primarily to conduct 
infrared astronomy. As the largest telescope in space, its greatly improved infrared 
resolution and sensitivity allow it to view objects too old, distant, or faint for the 
Hubble Space Telescope.
"""
summarize_and_measure(sample_text)

Loading phi3 and generating summary...

--- SUMMARY ---
The James Webb Space Telescope (JWST) is a highly advanced observatory aimed at conducting deep-space infrared astronomy from its position outside Earth'ries gravitational pull and atmospheric distortion. With superior resolution and sensitivity over the Hubble, JWST can observe some of the universe's most ancient and remote celestial objects in unprecedented detail, allowing for groundbreaking discoveries about the early cosmos.

📊 PERFORMANCE METRICS 📊
Total Time Taken:      1.10 seconds
Model Load Time:       0.03 seconds
Time to First Token:   0.13 seconds
----------------------------------------
Input Length:          98 tokens
Output Length:         101 tokens
Generation Speed:      115.30 tokens/second
----------------------------------------
Python RAM Increase:   0.06 MB
